# Analysis Notebook

Load saved numpy results and generate figures (no model loading).

In [ ]:

import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "results").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from viz_utils import (
    add_attention_entropy_overlay,
    add_dinov2_probe_backbone_separator,
    add_logit_corr_band,
    add_resnet_gradcam_target_marker,
    plot_cascade_paper_grid,
    plot_cascading_grid,
    select_depth_indices,
)

RESULTS_ROOT = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CLASS_A_METHODS = ["gradient", "smoothgrad", "input_grad", "ig"]
CLASS_B_METHODS = {
    "resnet50": ["gradcam"],
    "vit": ["transformer_gradcam"],
    "dinov2": ["transformer_gradcam"],
}
CLASS_C_METHODS = {
    "resnet50": ["gbp", "gbp_gc"],
    "vit": ["raw_attn", "rollout"],
    "dinov2": ["raw_attn", "rollout"],
}
ARCHS = {
    "resnet50": {"title": "ResNet-50"},
    "vit": {"title": "ViT-B/16"},
    "dinov2": {"title": "DINOv2-B/14"},
}
for arch in ARCHS:
    ARCHS[arch]["methods"] = (
        CLASS_A_METHODS + CLASS_B_METHODS[arch] + CLASS_C_METHODS[arch]
    )

MECH_ARCH_TAGS = [
    ("resnet", "ResNet-50"),
    ("vit", "ViT-B/16"),
    ("dinov2", "DINOv2-B/14"),
]
PRIMARY_SPEARMAN = {
    "ig": "spearman_rms_mean",
    "input_grad": "spearman_rms_mean",
}
DEFAULT_PRIMARY_SPEARMAN = "spearman_abs_rms_mean"
CLASS_B_SPATIAL = {
    "resnet50": "gradcam",
    "vit": "transformer_gradcam",
    "dinov2": "transformer_gradcam",
}


In [ ]:

def _load_order(arch):
    for base in [RESULTS_ROOT / arch, *sorted((RESULTS_ROOT / arch).glob("seed*"))]:
        order_path = base / "randomization_order.json"
        if order_path.exists():
            with open(order_path) as f:
                return json.load(f)
    return []


def _fraction_x(n_depths):
    if n_depths <= 1:
        return np.array([0.0])
    return np.linspace(0.0, 1.0, n_depths)


def _metric_suffix(method, metric="spearman"):
    if metric == "spearman":
        return PRIMARY_SPEARMAN.get(method, DEFAULT_PRIMARY_SPEARMAN)
    if method in ("ig", "input_grad"):
        return "ssim_rms_mean"
    return "ssim_abs_rms_mean"


def _method_dirs(arch, method):
    base = RESULTS_ROOT / arch
    if method in CLASS_A_METHODS:
        seed_dirs = sorted(base.glob("seed*"))
        if seed_dirs:
            return seed_dirs
    return [base]


def _load_curves(arch, method, metric="spearman"):
    suffix = _metric_suffix(method, metric=metric)
    curves = []
    for d in _method_dirs(arch, method):
        path = d / ("%s_%s.npy" % (method, suffix))
        if path.exists():
            curves.append(np.load(path))
    return curves


def _mean_std_curve(arch, method, metric="spearman"):
    curves = _load_curves(arch, method, metric=metric)
    if not curves:
        return None, None
    min_len = min(len(c) for c in curves)
    arr = np.stack([c[:min_len] for c in curves])
    return np.nanmean(arr, axis=0), np.nanstd(arr, axis=0)


def _plot_curve_with_band(ax, x, mean, std, label, **kwargs):
    ax.plot(x, mean, marker="o", markersize=3, label=label, **kwargs)
    if std is not None and np.any(std > 0):
        ax.fill_between(x, mean - std, mean + std, alpha=0.18)


def _apply_arch_overlays(ax, arch, order, x):
    add_logit_corr_band(ax, RESULTS_ROOT, arch, x_values=x)
    if arch == "resnet50":
        add_resnet_gradcam_target_marker(ax, order, x_values=x)
    if arch == "dinov2":
        add_dinov2_probe_backbone_separator(ax, x_values=x)


def _method_label(method):
    return method.replace("_", " ")


In [ ]:

def plot_within_architecture_curves(metric="spearman"):
    for arch, cfg in ARCHS.items():
        order = _load_order(arch)
        fig, ax = plt.subplots(figsize=(9, 4.5))
        for method in cfg["methods"]:
            mean, std = _mean_std_curve(arch, method, metric=metric)
            if mean is None:
                continue
            x = np.arange(len(mean))
            _plot_curve_with_band(ax, x, mean, std, _method_label(method))
        _apply_arch_overlays(ax, arch, order, np.arange(len(order)))
        ax.set_xlabel("Cascade depth")
        ax.set_ylabel("Primary %s" % metric)
        ax.set_title("%s within-architecture cascade curves" % cfg["title"])
        if order:
            ax.set_xticks(np.arange(len(order)))
            ax.set_xticklabels(order, rotation=60, ha="right", fontsize=7)
        ax.legend(fontsize=7, ncol=2)
        fig.tight_layout()
        out = FIGURES_DIR / ("within_arch_%s_%s.png" % (arch, metric))
        fig.savefig(out, dpi=150, bbox_inches="tight")
        plt.show()

plot_within_architecture_curves(metric="spearman")


In [ ]:

def plot_class_a_cross_architecture():
    for method in CLASS_A_METHODS:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
        for ax, arch in zip(axes, ARCHS):
            mean, std = _mean_std_curve(arch, method, metric="spearman")
            order = _load_order(arch)
            if mean is None:
                ax.set_title("%s (missing)" % ARCHS[arch]["title"])
                continue
            x = _fraction_x(len(mean))
            _plot_curve_with_band(ax, x, mean, std, ARCHS[arch]["title"])
            _apply_arch_overlays(ax, arch, order, x)
            ax.set_title(ARCHS[arch]["title"])
            ax.set_xlabel("Fractional cascade depth")
            ylabel = "Signed Spearman (RMS)" if method in ("ig", "input_grad") else "Abs Spearman (RMS)"
            ax.set_ylabel(ylabel)
        fig.suptitle("Class A portable method: %s" % _method_label(method), y=1.03)
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / ("class_a_%s_cross_arch.png" % method), dpi=150, bbox_inches="tight")
        plt.show()

plot_class_a_cross_architecture()


In [ ]:

def plot_class_b_spatial_attribution():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, arch in zip(axes, ARCHS):
        method = CLASS_B_SPATIAL[arch]
        mean, std = _mean_std_curve(arch, method, metric="spearman")
        order = _load_order(arch)
        if mean is None:
            ax.set_title("%s (missing)" % ARCHS[arch]["title"])
            continue
        x = _fraction_x(len(mean))
        _plot_curve_with_band(ax, x, mean, std, _method_label(method))
        _apply_arch_overlays(ax, arch, order, x)
        ax.set_title("%s: %s" % (ARCHS[arch]["title"], _method_label(method)))
        ax.set_xlabel("Fractional cascade depth")
        ax.set_ylabel("Abs Spearman (RMS)")
    fig.suptitle(
        "Architecture-native spatial attribution: GradCAM on ResNet layer4[-1]; transformer-adapted maps on blocks[11].norm2",
        y=1.04,
        fontsize=10,
    )
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "class_b_spatial_attribution.png", dpi=150, bbox_inches="tight")
    plt.show()


def plot_transformer_attention_diagnostics():
    for arch in ["vit", "dinov2"]:
        fig, ax = plt.subplots(figsize=(7, 4))
        x_values = None
        for method in ["raw_attn", "rollout"]:
            mean, std = _mean_std_curve(arch, method, metric="spearman")
            if mean is None:
                continue
            x_values = _fraction_x(len(mean))
            _plot_curve_with_band(ax, x_values, mean, std, _method_label(method))
        add_logit_corr_band(ax, RESULTS_ROOT, arch, x_values=x_values)
        add_attention_entropy_overlay(ax, RESULTS_ROOT / arch, num_patches=196 if arch == "vit" else 256, x_values=x_values)
        if arch == "dinov2":
            add_dinov2_probe_backbone_separator(ax, x_values=x_values)
        ax.set_title("%s attention diagnostics" % ARCHS[arch]["title"])
        ax.set_xlabel("Fractional cascade depth")
        ax.set_ylabel("Abs Spearman (RMS)")
        ax.legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / ("attention_diagnostics_%s.png" % arch), dpi=150, bbox_inches="tight")
        plt.show()

plot_class_b_spatial_attribution()
plot_transformer_attention_diagnostics()


In [ ]:

def _load_curve_stats(arch, method):
    stats = []
    for d in _method_dirs(arch, method):
        p = d / ("%s_curve_stats.json" % method)
        if p.exists():
            with open(p) as f:
                stats.append(json.load(f))
    return stats


def _fmt_mean_std(values, precision=3):
    vals = np.array([v for v in values if v is not None and np.isfinite(v)], dtype=float)
    if vals.size == 0:
        return "--"
    if vals.size == 1:
        return ("%%.%df" % precision) % vals[0]
    return (("%%.%df $\\pm$ %%.%df") % (precision, precision)) % (np.mean(vals), np.std(vals))


def build_sensitivity_ratio_table():
    rows = []
    for arch in ARCHS:
        for method in CLASS_A_METHODS:
            stats = _load_curve_stats(arch, method)
            if not stats:
                continue
            ratios = [s.get("sensitivity_ratio") for s in stats]
            ratio_mean = np.nanmean([r for r in ratios if r is not None])
            flag = ""
            if np.isfinite(ratio_mean) and ratio_mean > 1.5:
                flag = "potential failure"
            elif np.isfinite(ratio_mean) and ratio_mean < 0.5:
                flag = "potentially noise-driven"
            rows.append({
                "Architecture": ARCHS[arch]["title"],
                "Method Class": "A",
                "Method": method,
                "D_half": _fmt_mean_std([s.get("d_half") for s in stats]),
                "D_arch": _fmt_mean_std([s.get("d_arch") for s in stats]),
                "sensitivity_ratio": _fmt_mean_std(ratios),
                "final_sim": _fmt_mean_std([s.get("final_sim") for s in stats]),
                "normalized_auc": _fmt_mean_std([s.get("normalized_auc") for s in stats]),
                "Flag": flag,
            })
    return rows


def print_latex_table(rows):
    if not rows:
        print("No curve_stats JSON files found yet.")
        return
    cols = ["Architecture", "Method Class", "Method", "D_half", "D_arch", "sensitivity_ratio", "final_sim", "normalized_auc", "Flag"]
    line_end = r" \\"
    print(r"\begin{tabular}{lllllllll}")
    print(" & ".join(cols) + line_end)
    print(r"\hline")
    for row in rows:
        print(" & ".join(str(row[c]) for c in cols) + line_end)
    print(r"\end{tabular}")
    print("\nFootnote: DINOv2 sensitivity ratios exclude depth 0 (linear-probe-only randomization).")

sensitivity_rows = build_sensitivity_ratio_table()
print_latex_table(sensitivity_rows)


In [ ]:

mech = RESULTS_ROOT / "mechanistic"
if any((mech / ("logit_corr_%s.npy" % tag)).exists() for tag, _ in MECH_ARCH_TAGS):
    fig, ax = plt.subplots(figsize=(7, 4))
    for tag, label in MECH_ARCH_TAGS:
        p = mech / ("logit_corr_%s.npy" % tag)
        if p.exists():
            vals = np.load(p)
            ax.plot(_fraction_x(len(vals)), vals, marker="o", label=label)
    ax.set_xlabel("Fractional cascade depth")
    ax.set_ylabel("Mean logit Pearson r")
    ax.set_title("Model output preservation under cascading randomization")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "logit_correlation.png", dpi=150)
    plt.show()

for tag, label in MECH_ARCH_TAGS:
    files = sorted(mech.glob("activation_scale_%s_depth*.npy" % tag))
    if len(files) < 2:
        continue
    a0, a1 = np.load(files[0]), np.load(files[-1])
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(a0, bins=50, alpha=0.5, density=True, label="depth 0")
    ax.hist(a1, bins=50, alpha=0.5, density=True, label="depth %d" % (len(files) - 1))
    ax.set_title("Activation |.| scales: %s" % label)
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / ("activation_scales_%s.png" % tag), dpi=150)
    plt.show()


In [ ]:

for arch, cfg in ARCHS.items():
    qual_path = RESULTS_ROOT / arch / "qual_bundle.npz"
    if not qual_path.exists():
        print("Skip %s: no qual_bundle.npz (run qual-only Modal job or pipeline with skip_qual=False)" % arch)
        continue
    data = np.load(qual_path, allow_pickle=True)
    img_idx = int(data["image_index"]) if "image_index" in data.files else 0
    order = list(data["order"])
    methods = [m for m in cfg["methods"] if ("baseline_" + m) in data.files]
    depth_indices = select_depth_indices(len(order), max_cols=None)
    plot_cascade_paper_grid(
        qual_path,
        methods,
        depth_indices=depth_indices,
        out_path=FIGURES_DIR / ("cascade_grid_%s.png" % arch),
        title=cfg["title"] + " cascading randomization (image %d)" % img_idx,
        arch=arch,
        overlay=True,
        max_depth_cols=None,
        show=True,
    )
    plot_cascade_paper_grid(
        qual_path,
        methods,
        depth_indices=depth_indices,
        out_path=FIGURES_DIR / ("cascade_grid_%s_masks.png" % arch),
        title=cfg["title"] + " masks (image %d)" % img_idx,
        arch=arch,
        overlay=False,
        max_depth_cols=None,
        show=False,
    )
    if arch == "dinov2" and "dino_reference_attn" in data.files:
        print("DINOv2 dino_reference_attn shape:", data["dino_reference_attn"].shape)
    print("%s -> cascade_grid_%s.png (overlay) and cascade_grid_%s_masks.png (image %d)" % (arch, arch, arch, img_idx))

# Optional per-method vertical strips for slides
for arch, method, title in [
    ("resnet50", "gbp", "ResNet-50 GBP"),
    ("resnet50", "input_grad", "ResNet-50 Input-Grad"),
    ("resnet50", "ig", "ResNet-50 IG"),
    ("vit", "ig", "ViT IG"),
    ("vit", "transformer_gradcam", "ViT Transformer GradCAM"),
    ("vit", "raw_attn", "ViT raw attention"),
    ("dinov2", "ig", "DINOv2 IG"),
    ("dinov2", "transformer_gradcam", "DINOv2 Transformer GradCAM"),
]:
    plot_cascading_grid(
        RESULTS_ROOT / arch / "qual_bundle.npz",
        method,
        out_path=FIGURES_DIR / ("cascade_%s_%s.png" % (arch, method)),
        title=title,
        arch=arch,
        overlay=True,
        show=False,
    )

print("Figures saved to", FIGURES_DIR)
